## 0. Configuration & Namespace Setup
This initial block configures the physical and logical boundaries of the Lakehouse. 

* **Storage Paths:** Defines the connection to the Azure Data Lake Storage (ADLS Gen2) container.
* **State Management:** Establishes a dedicated `_checkpoints` directory, which is fundamentally required by Databricks Auto Loader to maintain stateful tracking of processed files.
* **Unity Catalog:** Explicitly routes all tables to the `dbw_proman_lakehouse_01.default` schema to ensure data governance and prevent tables from landing in the generic hive metastore.

In [0]:
from pyspark.sql import functions as F

STORAGE_ACCOUNT = "adlspromandev01"
CONTAINER = "datalake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CHECKPOINT_BASE = f"{BASE_PATH}/_checkpoints"

CATALOG = "dbw_proman_lakehouse_01"
SCHEMA = "default"
TARGET_SCHEMA = f"{CATALOG}.{SCHEMA}"

### 1. RDBMS Dimensions (Standard Batch Overwrite)
This block ingests the operational dimension tables (`users`, `projects`, `tasks`) exported as CSVs from the source RDBMS. 

* **Architectural Justification:** Because these relational exports act as complete daily snapshots reflecting the current state of the database, a standard batch `spark.read` combined with a full `mode("overwrite")` is the correct architectural pattern. 
* **Metadata Injection:** Automatically appends ingestion timestamps and the original ADLS file path for auditability.

In [0]:
for entity in ["users", "projects", "tasks"]:
    raw_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{BASE_PATH}/staging/rdbms/{entity}.csv")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    (
        raw_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TARGET_SCHEMA}.bronze_{entity}")
    )
    print(f"Registered {TARGET_SCHEMA}.bronze_{entity}")

Registered dbw_proman_lakehouse_01.default.bronze_users
Registered dbw_proman_lakehouse_01.default.bronze_projects
Registered dbw_proman_lakehouse_01.default.bronze_tasks


### 2. Time Tracking (Auto Loader Micro-Batch)
This block shifts from traditional batch processing to a stateful streaming paradigm using **Databricks Auto Loader** (`cloudFiles`) to process transactional IoT/event payloads.

* **Incremental State Management:** By utilizing a checkpoint directory, Auto Loader acts as a memory bank, instantly identifying and processing only the new JSON files that arrived since the previous execution, completely eliminating full table scans.
* **Cost Optimization:** The `trigger(availableNow=True)` command forces the stream to execute as a high-speed micro-batch. It processes the incremental queue and immediately shuts down the cluster compute, balancing streaming performance with batch-level cost efficiency.

In [0]:
print("Starting Auto Loader for Time-Tracking...")
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/schema_timetracking")
    .option("multiline", "true")
    .load(f"{BASE_PATH}/staging/timetracking/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze_timetracking")
    .trigger(availableNow=True)  
    .toTable(f"{TARGET_SCHEMA}.bronze_timetracking"))

Starting Auto Loader for Time-Tracking...


### 3. Document Manifests (Auto Loader Micro-Batch)
Similar to time-tracking, document and invoice metadata arrives as continuous JSON transactional events requiring incremental state tracking.

* **Schema Evolution Protection:** Leverages Auto Loader's `schemaLocation` to automatically infer and evolve the schema if the source system adds new metadata fields to the payload in the future.
* **Path Manipulation:** Extracts the underlying binary file location directly from the JSON manifest metadata using a regex replacement, constructing a clean `adls_raw_uri` for downstream audits.

In [0]:
print("Starting Auto Loader for Document Manifests...")
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json") 
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/schema_docs")
    .option("pathGlobFilter", "*.json")
    .option("multiline", "true")
    .load(f"{BASE_PATH}/staging/documents_raw/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn(
        "adls_raw_uri",
        F.regexp_replace(F.col("_metadata.file_path"), r"\.json$", "")
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze_docs")
    .trigger(availableNow=True)
    .toTable(f"{TARGET_SCHEMA}.bronze_documents_manifest"))

Starting Auto Loader for Document Manifests...
